# Fábrica de GIFs MapBiomas

Gere GIFs animados, colagens e imagens anuais de mapas do MapBiomas para **qualquer território**, sem programar.

**Como usar**
1. Rode a célula **1 · Conectar** (▶). Autorize o acesso ao Earth Engine e ao Google Drive quando o Colab pedir.
2. Rode a célula **2 · Abrir a Fábrica** e siga os 4 passos na tela: **Mapa → Território → Período e saídas → Gerar**.

Os arquivos vão para a pasta `MapBiomas_GIF_Factory` do seu Drive. O que já foi gerado não é refeito.


In [ ]:
#@title 1 · Conectar ao Earth Engine e ao Drive
#@markdown Projeto do Earth Engine que a sua conta pode usar:
GEE_PROJECT = 'mapbiomas-brazil'  #@param {type:"string"}
#@markdown Salvar os resultados no seu Google Drive:
USAR_DRIVE = True  #@param {type:"boolean"}
#@markdown ---
#@markdown Versão do código (só mude se alguém pedir):
REPO_REF = 'main'  #@param {type:"string"}

REPO_URL = 'https://github.com/mapbiomas/brazil-fire.git'
import os, subprocess, sys
from google.colab import drive, output
output.enable_custom_widget_manager()  # necessário para o mapa interativo

if not GEE_PROJECT.strip():
    raise ValueError('Preencha GEE_PROJECT com um projeto do Earth Engine.')

repo_dir = '/content/brazil-fire'
def git(*args, check=True):
    return subprocess.run(['git', '-C', repo_dir, *args], capture_output=True, text=True, check=check)

print('Baixando a Fábrica de GIFs…')
if not os.path.exists(repo_dir):
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, repo_dir], check=True)
git('fetch', '--prune', '--quiet', 'origin')
for ref in (f'origin/{REPO_REF}', REPO_REF):
    if git('checkout', '--quiet', '--detach', ref, check=False).returncode == 0:
        break
else:
    raise RuntimeError(f'Versão não encontrada: {REPO_REF}.')

package_dir = os.path.join(repo_dir, 'utils', 'mapbiomas_gif_factory')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', package_dir], check=True)
source_dir = os.path.join(package_dir, 'src')
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
for name in [m for m in sys.modules if m.split('.')[0] in {'gif_factory', 'mapbiomas_data'}]:
    del sys.modules[name]  # garante o código recém-baixado sem reiniciar o Colab

import ee
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)

if USAR_DRIVE:
    try:
        drive.mount('/content/drive')
    except Exception as exc:
        print(f'Drive não conectado ({exc}). Os arquivos ficarão só nesta sessão.')
print('✓ Pronto. Versão', git('rev-parse', '--short', 'HEAD').stdout.strip(), '· projeto', GEE_PROJECT)


In [ ]:
#@title 2 · Abrir a Fábrica
from gif_factory.app import show_app
fabrica = show_app(GEE_PROJECT)
